# **5. Preparar el entorno de recomendación**

In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

In [2]:
# Información descriptiva de las canciones
df = pd.read_csv('../data/spotify-tracks-dataset-clean.csv')

# Características musicales estandarizadas
X_audio_scaled = pd.read_csv('../data/spotify-audio-features-scaled.csv', index_col = 0)

print('Dimensiones de df:', df.shape)
print('Dimensiones de X_audio_scaled:', X_audio_scaled.shape)

Dimensiones de df: (89578, 20)
Dimensiones de X_audio_scaled: (89578, 9)


In [3]:
print('¿Tienen los mismos índices?', df.index.equals(X_audio_scaled.index))
print('¿Tienen la misma cantidad de canciones?', len(df) == len(X_audio_scaled))

¿Tienen los mismos índices? True
¿Tienen la misma cantidad de canciones? True


### **5.1 Cálculo de las similitudes de una canción**

### **5.1.1 similitud del coseno**

In [4]:
# Seleccionar el índice de la primera canción
idx_ref = X_audio_scaled.index[0]

# Recuperar sus características musicales
vector_ref = X_audio_scaled.loc[[idx_ref]]

print('Índice de referencia:', idx_ref)
print('Dimensiones del vector:', vector_ref.shape)

Índice de referencia: 0
Dimensiones del vector: (1, 9)


In [5]:
# Comparar la canción de referencia con todas las canciones
similarities = cosine_similarity(vector_ref, X_audio_scaled)

print('Dimensiones del resultado:', similarities.shape)

Dimensiones del resultado: (1, 89578)


In [6]:
similarities = pd.Series(similarities.ravel(), index = X_audio_scaled.index)

print('Similitud con la propia canción:', similarities.loc[idx_ref])

print('\nSimilitud mínima:', similarities.min())
print('Similitud máxima:', similarities.max())
print('Cantidad de comparaciones:', len(similarities))

Similitud con la propia canción: 1.0

Similitud mínima: -0.9135408128624746
Similitud máxima: 1.0
Cantidad de comparaciones: 89578


In [7]:
# Ordenar las similitudes de mayor a menor
similarities_sorted = similarities.sort_values(ascending = False)

# Excluir la canción de referencia
similarities_sorted = similarities_sorted.drop(idx_ref)

# Mostrar las primeras 10 coincidencias
similarities_sorted.head(10)

12098    0.986727
72671    0.985363
59169    0.984864
23598    0.975005
23586    0.975005
1143     0.974303
45558    0.973923
46156    0.973870
38828    0.969403
73154    0.968434
dtype: float64

In [8]:
# Obtener los índices de las 10 canciones más similares
top_indices = similarities_sorted.head(10).index

# Recuperar su información descriptiva
recommendations = df.loc[top_indices, ['track_name', 'artists', 'album_name']].copy()

# Incorporar los valores de similitud
recommendations['similarity'] = similarities_sorted.loc[top_indices]

recommendations

,track_name,artists,album_name,similarity
12098,Infatuation,SOPHIE,OIL OF EVERY PEARL'S UN-INSIDES,0.986727
72671,How Great Is Your Love,Phil Wickham,Living Hope (The House Sessions),0.985363
59169,Feel Again (Feat. Au/Ra),Kina;Au/Ra,Things I Wanted To Tell You,0.984864
23598,Tourner Dans Le Vide - Version Orchestrale,Indila,Best Viral Tunes,0.975005
23586,Tourner Dans Le Vide - Version Orchestrale,Indila,Viral Şarkılar,0.975005
1143,Llueve,Jorge Drexler,Llueve,0.974303
45558,Daiva Pithave,Don Valiyavelicham;Aju Philip,Lord Saviour,0.973923
46156,醉梦前尘,林志炫,魔道祖师动画原声带,0.973870
38828,Say the Name,clipping.,Visions of Bodies Being Burned,0.969403
73154,You Saved My Soul,Bryan & Katie Torwalt,Here On Earth,0.968434


In [9]:
df.loc[idx_ref, ['track_name', 'artists', 'album_name']]

track_name            To Begin Again
artists       Ingrid Michaelson;ZAYN
album_name            To Begin Again
Name: 0, dtype: str

In [10]:
# Índices de las cinco recomendaciones principales
top5_indices = similarities_sorted.head(5).index

# Recuperar las características de referencia y recomendaciones
comparison = X_audio_scaled.loc[[idx_ref] + list(top5_indices)].copy()

# Incorporar nombres de canciones
comparison.insert(0, 'track_name', df.loc[comparison.index, 'track_name'])

comparison

,track_name,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,To Begin Again,-0.714105,-1.080798,-0.243196,-0.281477,-0.348924,-0.534101,-0.512292,-1.335230,-1.546444
12098,Infatuation,-1.187664,-1.174654,-0.374518,-0.457969,-0.393289,-0.469725,-0.486565,-1.642065,-1.607002
72671,How Great Is Your Love,-0.696988,-0.830515,-0.212103,-0.516211,-0.440612,-0.534101,-0.785518,-1.400028,-1.628310
59169,Feel Again (Feat. Au/Ra),-0.434534,-0.983031,-0.409280,-0.255004,-0.260193,-0.534101,-0.476274,-0.938822,-1.485650
23598,Tourner Dans Le Vide - Version Orchestrale,-1.336008,-1.069066,-0.325851,-0.486207,-0.360755,-0.534101,-0.337345,-1.396216,-2.111427
23586,Tourner Dans Le Vide - Version Orchestrale,-1.336008,-1.069066,-0.325851,-0.486207,-0.360755,-0.534101,-0.337345,-1.396216,-2.111427


### **5.1.2 Distancias euclidianas**

In [11]:
# Calcular distancias euclidianas
distances = euclidean_distances(vector_ref, X_audio_scaled)

# Convertir el resultado en una Serie
distances = pd.Series(distances.ravel(), index = X_audio_scaled.index)

# Ordenar de menor a mayor
distances_sorted = distances.sort_values()

# Excluir la canción de referencia
distances_sorted = distances_sorted.drop(idx_ref)

distances_sorted.head(10)

72671    0.461477
59169    0.534798
45558    0.587550
12098    0.621330
38828    0.638849
1143     0.663536
29973    0.666087
64947    0.670502
22376    0.675701
73154    0.679563
dtype: float64

In [12]:
# Obtener los índices de las 10 canciones más cercanas
top_indices_euclidean = distances_sorted.head(10).index

# Recuperar información descriptiva
recommendations_euclidean = df.loc[top_indices_euclidean, ['track_name', 'artists', 'album_name']].copy()

# Incorporar las distancias
recommendations_euclidean['distance'] = (distances_sorted.loc[top_indices_euclidean])

recommendations_euclidean

,track_name,artists,album_name,distance
72671,How Great Is Your Love,Phil Wickham,Living Hope (The House Sessions),0.461477
59169,Feel Again (Feat. Au/Ra),Kina;Au/Ra,Things I Wanted To Tell You,0.534798
45558,Daiva Pithave,Don Valiyavelicham;Aju Philip,Lord Saviour,0.587550
12098,Infatuation,SOPHIE,OIL OF EVERY PEARL'S UN-INSIDES,0.621330
38828,Say the Name,clipping.,Visions of Bodies Being Burned,0.638849
1143,Llueve,Jorge Drexler,Llueve,0.663536
29973,Linda Mi Cholita,William Luna,Mírame,0.666087
64947,You & I (Nobody in the World),John Legend,Love In The Future (Expanded Edition),0.670502
22376,Atlantis - Slowed Down Version,Seafret,Atlantis,0.675701
73154,You Saved My Soul,Bryan & Katie Torwalt,Here On Earth,0.679563


In [13]:
# Índices de las cinco recomendaciones principales
top5_indices = distances_sorted.head(5).index

# Recuperar las características de referencia y recomendaciones
comparison = X_audio_scaled.loc[[idx_ref] + list(top5_indices)].copy()

# Incorporar nombres de canciones
comparison.insert(0, 'track_name', df.loc[comparison.index, 'track_name'])

comparison

,track_name,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,To Begin Again,-0.714105,-1.080798,-0.243196,-0.281477,-0.348924,-0.534101,-0.512292,-1.335230,-1.546444
72671,How Great Is Your Love,-0.696988,-0.830515,-0.212103,-0.516211,-0.440612,-0.534101,-0.785518,-1.400028,-1.628310
59169,Feel Again (Feat. Au/Ra),-0.434534,-0.983031,-0.409280,-0.255004,-0.260193,-0.534101,-0.476274,-0.938822,-1.485650
45558,Daiva Pithave,-0.919504,-0.736658,-0.373552,-0.505621,-0.470189,-0.534101,-0.440255,-1.026489,-1.592763
12098,Infatuation,-1.187664,-1.174654,-0.374518,-0.457969,-0.393289,-0.469725,-0.486565,-1.642065,-1.607002
38828,Say the Name,-0.617111,-0.818783,-0.179852,-0.353839,-0.461316,-0.273193,-0.249871,-1.640159,-1.266679


### **5.2 Comparación cuantitativa de los rankings**

In [14]:
# Obtener posiciones de cada canción en ambos métodos
rank_cosine = similarities_sorted.rank(ascending = False, method = 'min')
rank_euclidean = distances_sorted.rank(ascending = True, method = 'min')

# Identificar las canciones que aparecen en alguno de los dos top 10
top10_union = (similarities_sorted.head(10).index.union(distances_sorted.head(10).index))

# Construir tabla comparativa
ranking_comparison = pd.DataFrame({
    'track_name': df.loc[top10_union, 'track_name'],
    'artists': df.loc[top10_union, 'artists'],
    'rank_cosine': rank_cosine.loc[top10_union],
    'rank_euclidean': rank_euclidean.loc[top10_union]
})

ranking_comparison.sort_values('rank_cosine')

,track_name,artists,rank_cosine,rank_euclidean
12098,Infatuation,SOPHIE,1.0,4.0
72671,How Great Is Your Love,Phil Wickham,2.0,1.0
59169,Feel Again (Feat. Au/Ra),Kina;Au/Ra,3.0,2.0
23586,Tourner Dans Le Vide - Version Orchestrale,Indila,4.0,65.0
23598,Tourner Dans Le Vide - Version Orchestrale,Indila,4.0,65.0
1143,Llueve,Jorge Drexler,6.0,6.0
45558,Daiva Pithave,Don Valiyavelicham;Aju Philip,7.0,3.0
46156,醉梦前尘,林志炫,8.0,21.0
38828,Say the Name,clipping.,9.0,5.0
73154,You Saved My Soul,Bryan & Katie Torwalt,10.0,10.0


### **5.2.1 Comparación de magnitudes y similitudes**
* `To Begin Again`: referencia.
* `Tourner Dans Le Vide`: una de las canciones que presenta la diferencia más llamativa.
* `Infatuation`: primera recomendación del coseno.
* `How Great Is Your Love`: primera recomendación euclidiana.

In [15]:
# Canciones que queremos examinar
indices_test = [0, 23598, 12098, 72671]

# Construir tabla comparativa
geometry_comparison = pd.DataFrame(index = indices_test)

geometry_comparison['track_name'] = (df.loc[indices_test, 'track_name'])

# Magnitud de cada vector
geometry_comparison['magnitude'] = [
    np.linalg.norm(X_audio_scaled.loc[idx].values)
    for idx in indices_test
]

# Similitud respecto a la canción de referencia
geometry_comparison['cosine_similarity'] = (similarities.loc[indices_test])

# Distancia respecto a la canción de referencia
geometry_comparison['euclidean_distance'] = (distances.loc[indices_test])

geometry_comparison

,track_name,magnitude,cosine_similarity,euclidean_distance
0,To Begin Again,2.580742,1.000000,0.000000
23598,Tourner Dans Le Vide - Version Orchestrale,3.194863,0.975005,0.888436
12098,Infatuation,3.005204,0.986727,0.621330
72671,How Great Is Your Love,2.682335,0.985363,0.461477


### **5.3 Creación del recomendador**

### **5.3.1 Construcción del buscador de canciones**

In [16]:
# Definir los criterios de búsqueda
nombre = "don't Stop Me No "
artista = ' quee'

# Buscar coincidencias por título
mascara_nombre = df['track_name'].str.contains(nombre.strip(), case = False, na = False, regex = False)

# Buscar coincidencias por artista
mascara_artista = df['artists'].str.contains(artista.strip(), case = False, na = False, regex = False)

# Combinar ambas condiciones
coincidencias = df[mascara_nombre & mascara_artista]

# Mostrar resultados
display(coincidencias[['track_name', 'artists', 'album_name']])

,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)


### **5.3.2 Coincidencias múltiples**

In [17]:
nombre = 'LOve'
artista = None

mascara_nombre = df['track_name'].str.contains(nombre.strip(), case = False, na = False, regex = False)

coincidencias = df[mascara_nombre]

print('Número de coincidencias:', len(coincidencias))
display(coincidencias[['track_name', 'artists', 'album_name']].head(10))

Número de coincidencias: 2091


,track_name,artists,album_name
1,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...
7,Falling in Love at a Coffee Shop,Landon Pigg,The Boy Who Never
20,And I Love Her,Kurt Cobain,Montage Of Heck: The Home Recordings
40,Can’t Help Falling in Love,Boyce Avenue,"Cover Sessions, Vol. 6"
46,I Was Born To Love You,Ray LaMontagne;Sierra Ferrell,I Was Born To Love You
57,Please Don't Say You Love Me,Gabrielle Aplin,English Rain
61,Make You Feel My Love,JJ Heller,I Dream of You: CALM
65,"On Love, in Sadness",Jason Mraz,Waiting for My Rocket to Come (Expanded Edition)
84,Love Song,Sara Bareilles,Little Voice
85,Love Story,Tyler Ward;Karis;Ray Lorraine,Love Story


In [18]:
nombre = 'LOve'
artista = ' Jason Mraz'

# Máscara para el nombre
mascara_nombre = df['track_name'].str.contains(nombre.strip(), case = False, na = False, regex = False)

# Máscara para el artista
mascara_artista = df['artists'].str.contains(artista.strip(), case = False, na = False, regex = False)

# Combinar condiciones
coincidencias = df[mascara_nombre & mascara_artista]

print('Número de coincidencias:', len(coincidencias))
display(coincidencias[['track_name', 'artists', 'album_name']])

Número de coincidencias: 4


,track_name,artists,album_name
65,"On Love, in Sadness",Jason Mraz,Waiting for My Rocket to Come (Expanded Edition)
155,Love Someone,Jason Mraz,YES!
509,The Woman I Love,Jason Mraz,Love Is a Four Letter Word
38111,Could I Love You Any More (feat. Jason Mraz),Reneé Dominique;Jason Mraz,Could I Love You Any More (feat. Jason Mraz)


### **5.4 Construcción de la función**

In [19]:
def buscar_cancion(nombre, artista = None, limite = 20):
    
    # Buscar coincidencias en el título
    mascara_nombre = df['track_name'].str.contains(nombre.strip(), case = False, na = False, regex = False)

    # Si se proporciona un artista, aplicar también ese filtro
    if artista is not None:
        mascara_artista = df['artists'].str.contains(artista.strip(), case = False, na = False, regex = False)

        coincidencias = df[mascara_nombre & mascara_artista]

    else:
        coincidencias = df[mascara_nombre]

    # Evaluar el número de coincidencias
    numero = len(coincidencias)

    if numero == 0:
        print('No se encontraron coincidencias.')

    elif numero == 1:
        print('Se encontró una coincidencia:')
        display(coincidencias[['track_name', 'artists', 'album_name']])

    else:
        print(f'Se encontraron {numero} coincidencias.')

        if numero > limite:
            print(f'Mostrando las primeras {limite} coincidencias.')

        display(coincidencias[['track_name', 'artists', 'album_name']].head(limite))

    return coincidencias

### **5.4.1 Búsqueda por título**

In [20]:
coincidencias = buscar_cancion(
    nombre = "LOve "
)

Se encontraron 2091 coincidencias.
Mostrando las primeras 20 coincidencias.


,track_name,artists,album_name
1,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...
7,Falling in Love at a Coffee Shop,Landon Pigg,The Boy Who Never
20,And I Love Her,Kurt Cobain,Montage Of Heck: The Home Recordings
40,Can’t Help Falling in Love,Boyce Avenue,"Cover Sessions, Vol. 6"
46,I Was Born To Love You,Ray LaMontagne;Sierra Ferrell,I Was Born To Love You
57,Please Don't Say You Love Me,Gabrielle Aplin,English Rain
61,Make You Feel My Love,JJ Heller,I Dream of You: CALM
65,"On Love, in Sadness",Jason Mraz,Waiting for My Rocket to Come (Expanded Edition)
84,Love Song,Sara Bareilles,Little Voice
85,Love Story,Tyler Ward;Karis;Ray Lorraine,Love Story


### **5.4.2 Búsqueda por título y artista**

In [21]:
coincidencias = buscar_cancion(
    nombre = "LOve ",
    artista = " Jason Mraz"
)

Se encontraron 4 coincidencias.


,track_name,artists,album_name
65,"On Love, in Sadness",Jason Mraz,Waiting for My Rocket to Come (Expanded Edition)
155,Love Someone,Jason Mraz,YES!
509,The Woman I Love,Jason Mraz,Love Is a Four Letter Word
38111,Could I Love You Any More (feat. Jason Mraz),Reneé Dominique;Jason Mraz,Could I Love You Any More (feat. Jason Mraz)


### **5.4.3 Sin coincidencias**

In [22]:
coincidencias = buscar_cancion(
    nombre = "Esta canción no existe 12345"
)

No se encontraron coincidencias.


### **5.4.4 Una coincidencia**

In [23]:
coincidencias = buscar_cancion(
    nombre = "Don't Stop Me Now - Remastered 2011"
)

Se encontró una coincidencia:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)


### **5.4.5 Selección de canciones**

In [24]:
# Seleccionar el índice de la canción
indice = 155

# Recuperar sus metadatos
cancion = df.loc[indice]

# Recuperar sus características de audio escaladas
caracteristicas = X_audio_scaled.loc[indice]

print('Canción seleccionada:')
display(cancion[['track_name', 'artists', 'album_name']])

print('Características de audio:')
display(caracteristicas)

Canción seleccionada:


track_name    Love Someone
artists         Jason Mraz
album_name            YES!
Name: 155, dtype: object

Características de audio:


danceability        0.107491
energy             -1.127726
loudness           -1.215559
speechiness        -0.503856
acousticness       -0.032451
instrumentalness   -0.513117
liveness           -0.543165
valence            -1.488838
tempo              -0.818503
Name: 155, dtype: float64

### **5.4.6 Selección de una canción entre varias coincidencias**

In [25]:
# Buscar canciones
coincidencias = buscar_cancion(
    nombre = 'Love',
    artista = 'Jason Mraz'
)

# Seleccionar una canción
if len(coincidencias) > 0:

    indice = int(input('Introduce el índice de la canción que deseas seleccionar: '))

    if indice in coincidencias.index:

        cancion = df.loc[indice]
        caracteristicas = X_audio_scaled.loc[indice]

        print('\nCanción seleccionada:')
        display(cancion[['track_name', 'artists', 'album_name']])

    else:
        print('El índice introducido no corresponde a ninguna coincidencia.')

Se encontraron 4 coincidencias.


,track_name,artists,album_name
65,"On Love, in Sadness",Jason Mraz,Waiting for My Rocket to Come (Expanded Edition)
155,Love Someone,Jason Mraz,YES!
509,The Woman I Love,Jason Mraz,Love Is a Four Letter Word
38111,Could I Love You Any More (feat. Jason Mraz),Reneé Dominique;Jason Mraz,Could I Love You Any More (feat. Jason Mraz)



Canción seleccionada:


track_name    Love Someone
artists         Jason Mraz
album_name            YES!
Name: 155, dtype: object

### **5.5 Estructura del proceso de selección**

### **5.5.1 Buscar canción (reutilizada)**

In [26]:
def buscar_cancion(nombre, artista = None, limite = 20):
    
    # Buscar coincidencias en el título
    mascara_nombre = df['track_name'].str.contains(nombre.strip(), case = False, na = False, regex = False)

    # Si se proporciona un artista, aplicar también ese filtro
    if artista is not None:
        mascara_artista = df['artists'].str.contains(artista.strip(), case = False, na = False, regex = False)

        coincidencias = df[mascara_nombre & mascara_artista]

    else:
        coincidencias = df[mascara_nombre]

    # Evaluar el número de coincidencias
    numero = len(coincidencias)

    if numero == 0:
        print('No se encontraron coincidencias.')

    elif numero == 1:
        print('Se encontró una coincidencia:')
        display(coincidencias[['track_name', 'artists', 'album_name']])

    else:
        print(f'Se encontraron {numero} coincidencias.')

        if numero > limite:
            print(f'Mostrando las primeras {limite} coincidencias.')

        display(coincidencias[['track_name', 'artists', 'album_name']].head(limite))

    return coincidencias

### **5.5.2 Seleccionar canción**

In [ ]:
def seleccionar_cancion():
    
    while True:

        # Solicitar los datos de búsqueda
        nombre = input('Introduce el nombre de la canción (o "salir" para terminar): ')

        # Comprobar si el usuario desea salir
        if nombre.strip().lower() == 'salir':
            print('Búsqueda finalizada.')
            return None

        # Comprobar que el nombre no esté vacío
        if not nombre.strip():
            print('Debes introducir el nombre de una canción.\n')
            continue

        artista = input('Introduce el artista (opcional): ')

        # Buscar coincidencias
        if artista.strip():
            coincidencias = buscar_cancion(nombre, artista)
        else:
            coincidencias = buscar_cancion(nombre)

        # Comprobar si existen coincidencias
        if coincidencias.empty:
            print('\nIntenta realizar una nueva búsqueda.\n')
            continue

        # Si existe una sola coincidencia, obtener su índice
        if len(coincidencias) == 1:
            indice = coincidencias.index[0]

        # Si existen varias coincidencias, solicitar una selección
        else:
            # Limitar las opciones a las primeras 20 coincidencias
            opciones = coincidencias.head(20)

            print('\nIntroduce el índice de la canción que deseas seleccionar.')

            while True:
                entrada = input('Índice: ')

                try:
                    indice = int(entrada)

                    if indice in opciones.index:
                        break
                    else:
                        print('El índice no corresponde a ninguna de las opciones mostradas.')

                except ValueError:
                    print('Debes introducir un índice numérico.')

        # Mostrar la canción seleccionada
        print('\nCanción seleccionada:')
        display(df.loc[[indice], ['track_name', 'artists', 'album_name']])

        # Solicitar confirmación
        while True:
            respuesta = input('¿Es esta la canción que buscabas? (sí/no): ')
            respuesta = respuesta.strip().lower()

            if respuesta in ['si', 'sí']:
                print('\nCanción confirmada.')
                return indice

            elif respuesta == 'no':
                print('\nVamos a realizar una nueva búsqueda.\n')
                break

            else:
                print('Respuesta no válida. Introduce "sí" o "no".')

In [50]:
indice = seleccionar_cancion()

if indice is not None:
    cancion = df.loc[indice]
    caracteristicas = X_audio_scaled.loc[indice]

Se encontró una coincidencia:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)



Canción seleccionada:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)



Canción confirmada.


### **5.6 Implementación**

### **5.6.1 Seleccionar canción (reutilizada)**

In [52]:
indice = seleccionar_cancion()

if indice is not None:
    cancion = df.loc[indice]
    caracteristicas = X_audio_scaled.loc[[indice]]

Se encontró una coincidencia:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)



Canción seleccionada:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)



Canción confirmada.


### **5.7 Prueba**

In [64]:
if indice is not None:
    
    similitudes = cosine_similarity(caracteristicas, X_audio_scaled)

    similitudes = pd.Series(
        similitudes.flatten(),
        index = X_audio_scaled.index
    )

    similitudes = similitudes.drop(index = indice)

if indice is not None:
    
    print('Canción de referencia:')
    display(df.loc[[indice], ['track_name', 'artists', 'album_name']])

    print('\nPrimeras 10 similitudes:')
    display(similitudes.sort_values(ascending = False).head(10))

Canción de referencia:


,track_name,artists,album_name
57411,Don't Stop Me Now - Remastered 2011,Queen,Jazz (2011 Remaster)



Primeras 10 similitudes:


80872    0.994658
31521    0.990451
40826    0.989741
23216    0.987003
29497    0.986651
33391    0.986361
41972    0.985956
61615    0.983264
6815     0.981435
29627    0.978996
dtype: float64

### **5.7.1 Recomendaciones**

In [65]:
if indice is not None:
    
    indices_recomendaciones = similitudes.sort_values(ascending = False).head(10).index

    recomendaciones = df.loc[indices_recomendaciones, ['track_name', 'artists', 'album_name']].copy()

    recomendaciones['cosine_similarity'] = similitudes.loc[indices_recomendaciones]

    display(recomendaciones)

,track_name,artists,album_name,cosine_similarity
80872,Spotlight,Marshmello;Lil Peep,Spotlight,0.994658
31521,All That I Need (feat. Kayliana & MC Riddle),S3RL;Kayliana;MC Riddle,All That I Need (feat. Kayliana & MC Riddle),0.990451
40826,仮契約のシンデレラ (中吉 ver.),Shiritsu Ebisu Chugaku,Major Debut 10th Anniversary Album 中吉,0.989741
23216,Vou Te Amar o Ano Inteiro - Ao Vivo,Bell Marques,Vou Te Amar o Ano Inteiro (Ao Vivo),0.987003
29497,Wastelands,Linkin Park,The Hunting Party,0.986651
33391,Blood Of Humanity,Ncrypta,Supremacy 2022 (The Nation Of Supreme),0.986361
41972,ギター,Humbreaders,ギター,0.985956
61615,Das Duas Uma - Ao Vivo,Júnior e Cézar,Das Duas Uma (Ao Vivo),0.983264
6815,Abba Zabba,CIRRUS,Back on a Mission,0.981435
29627,Ppr:kut (Cheapshot and Jubacca Reanimation) [f...,Linkin Park;Planet Asia;Rasco,Reanimation,0.978996


### **5.7.2 Análisis de las recomendaciones**

In [66]:
if indice is not None:
    
    columnas_audio = [
        'danceability',
        'energy',
        'loudness',
        'speechiness',
        'acousticness',
        'instrumentalness',
        'liveness',
        'valence',
        'tempo'
    ]

    indices_comparacion = [indice] + list(indices_recomendaciones)

    comparacion = df.loc[indices_comparacion, ['track_name', 'artists'] + columnas_audio]

    display(comparacion)

,track_name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
57411,Don't Stop Me Now - Remastered 2011,Queen,0.559,0.868,-5.276,0.170,0.047500,0.000176,0.776,0.609,156.295
80872,Spotlight,Marshmello;Lil Peep,0.546,0.822,-4.483,0.151,0.014500,0.000000,0.742,0.615,150.060
31521,All That I Need (feat. Kayliana & MC Riddle),S3RL;Kayliana;MC Riddle,0.581,0.949,-3.717,0.159,0.000766,0.000000,0.894,0.730,174.990
40826,仮契約のシンデレラ (中吉 ver.),Shiritsu Ebisu Chugaku,0.520,0.960,-3.113,0.138,0.020100,0.000063,0.929,0.664,170.047
23216,Vou Te Amar o Ano Inteiro - Ao Vivo,Bell Marques,0.490,0.985,-2.831,0.165,0.029000,0.000000,0.984,0.547,167.679
29497,Wastelands,Linkin Park,0.440,0.993,-3.481,0.184,0.010400,0.000000,0.964,0.614,169.770
33391,Blood Of Humanity,Ncrypta,0.553,0.939,-2.380,0.159,0.028800,0.043900,0.814,0.695,159.966
41972,ギター,Humbreaders,0.486,0.941,-2.875,0.156,0.005040,0.000000,0.811,0.645,163.866
61615,Das Duas Uma - Ao Vivo,Júnior e Cézar,0.593,0.833,-4.806,0.109,0.136000,0.000000,0.726,0.606,157.859
6815,Abba Zabba,CIRRUS,0.595,0.963,-5.035,0.146,0.175000,0.008960,0.961,0.725,169.718


In [67]:
if indice is not None:
    
    diferencias = X_audio_scaled.loc[indices_recomendaciones].sub(X_audio_scaled.loc[indice].iloc[0]).abs()

    diferencias.index = indices_recomendaciones

    display(diferencias)

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
80872,0.074172,0.753581,0.794618,0.583240,0.903416,0.510365,2.727384,0.575255,0.959125
31521,0.125522,1.250237,0.942548,0.653836,0.944037,0.510365,3.509500,1.013591,1.798319
40826,0.222516,1.293254,1.059193,0.468520,0.886853,0.510170,3.689593,0.762025,1.631928
23216,0.393681,1.391021,1.113653,0.706784,0.860529,0.510365,3.972596,0.316066,1.552216
29497,0.678958,1.422307,0.988125,0.874450,0.915542,0.510365,3.869686,0.571444,1.622604
33391,0.034233,1.211130,1.200751,0.653836,0.861121,0.374495,3.097860,0.880185,1.292581
41972,0.416504,1.218952,1.105156,0.627363,0.931395,0.510365,3.082424,0.689604,1.423863
61615,0.193988,0.796599,0.732240,0.212608,0.544057,0.510365,2.645056,0.540951,1.221655
6815,0.205399,1.304986,0.688015,0.539117,0.428707,0.482634,3.854249,0.994533,1.620853
29627,0.034233,1.046882,0.440048,0.230257,0.944613,0.507645,2.233415,0.521893,0.969089


In [69]:
if indice is not None:
    
    vector_referencia = X_audio_scaled.loc[indice].to_numpy()

    productos = X_audio_scaled.loc[
        indices_recomendaciones
    ].mul(vector_referencia, axis = 'columns')

    normas = np.linalg.norm(
        X_audio_scaled.loc[indices_recomendaciones].to_numpy(),
        axis = 1
    )

    contribuciones = productos.div(
        np.linalg.norm(vector_referencia) * pd.Series(normas, index=indices_recomendaciones),
        axis = 0
    )

    print('Contribuciones de cada característica a la similitud coseno:')
    display(contribuciones)

Contribuciones de cada característica a la similitud coseno:


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
80872,0.000197,0.056211,0.040315,0.034444,0.065113,0.024126,0.658877,0.024683,0.090693
31521,-0.000151,0.069872,0.035542,0.028692,0.050273,0.017845,0.628341,0.032769,0.127269
40826,0.000363,0.071672,0.039695,0.020072,0.046877,0.017678,0.654865,0.024221,0.114299
23216,0.000585,0.073451,0.039758,0.029330,0.043316,0.016828,0.671239,0.009126,0.103370
29497,0.000982,0.074905,0.035073,0.036419,0.045872,0.016777,0.651773,0.017046,0.107804
33391,0.000097,0.076278,0.051342,0.032354,0.051832,0.015004,0.624869,0.031971,0.102613
41972,0.000734,0.076345,0.046905,0.030819,0.055632,0.020009,0.618208,0.024716,0.112589
61615,-0.000354,0.061610,0.038351,0.012035,0.041273,0.024971,0.661201,0.023959,0.120218
6815,-0.000265,0.071673,0.025233,0.023045,0.023079,0.016613,0.678025,0.031558,0.112474
29627,-0.000024,0.089941,0.024850,0.014511,0.077621,0.027396,0.614633,0.025447,0.104619


In [70]:
if indice is not None:
    
    print('Características estandarizadas de la canción de referencia:')

    display(
        X_audio_scaled.loc[indice].to_frame('valor_estandarizado')
    )

    print('Estadísticas originales de liveness:')

    display(
        df['liveness'].describe()
    )

Características estandarizadas de la canción de referencia:


,valor_estandarizado
danceability,-0.023736
energy,0.909736
loudness,0.617737
speechiness,0.727170
acousticness,-0.829548
instrumentalness,-0.533556
liveness,2.878595
valence,0.528650
tempo,1.145272


Estadísticas originales de liveness:


count    89578.000000
mean         0.216561
std          0.194346
min          0.009250
25%          0.098200
50%          0.132000
75%          0.278000
max          1.000000
Name: liveness, dtype: float64

In [72]:
if indice is not None:
    
    vector_referencia = X_audio_scaled.loc[indice]

    productos = X_audio_scaled.loc[
        indices_recomendaciones
    ].mul(vector_referencia)

    denominador = (
        np.linalg.norm(vector_referencia)
        * np.linalg.norm(X_audio_scaled.loc[indices_recomendaciones], axis = 1)
    )

    contribuciones_verificadas = productos.div(denominador, axis = 0)

    display(contribuciones_verificadas)

    print('Suma de contribuciones:')
    display(contribuciones_verificadas.sum(axis=1))

    print('Similitudes originales:')
    display(similitudes.loc[indices_recomendaciones])

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
80872,0.000197,0.056211,0.040315,0.034444,0.065113,0.024126,0.658877,0.024683,0.090693
31521,-0.000151,0.069872,0.035542,0.028692,0.050273,0.017845,0.628341,0.032769,0.127269
40826,0.000363,0.071672,0.039695,0.020072,0.046877,0.017678,0.654865,0.024221,0.114299
23216,0.000585,0.073451,0.039758,0.029330,0.043316,0.016828,0.671239,0.009126,0.103370
29497,0.000982,0.074905,0.035073,0.036419,0.045872,0.016777,0.651773,0.017046,0.107804
33391,0.000097,0.076278,0.051342,0.032354,0.051832,0.015004,0.624869,0.031971,0.102613
41972,0.000734,0.076345,0.046905,0.030819,0.055632,0.020009,0.618208,0.024716,0.112589
61615,-0.000354,0.061610,0.038351,0.012035,0.041273,0.024971,0.661201,0.023959,0.120218
6815,-0.000265,0.071673,0.025233,0.023045,0.023079,0.016613,0.678025,0.031558,0.112474
29627,-0.000024,0.089941,0.024850,0.014511,0.077621,0.027396,0.614633,0.025447,0.104619


Suma de contribuciones:


80872    0.994658
31521    0.990451
40826    0.989741
23216    0.987003
29497    0.986651
33391    0.986361
41972    0.985956
61615    0.983264
6815     0.981435
29627    0.978996
dtype: float64

Similitudes originales:


80872    0.994658
31521    0.990451
40826    0.989741
23216    0.987003
29497    0.986651
33391    0.986361
41972    0.985956
61615    0.983264
6815     0.981435
29627    0.978996
dtype: float64